# Intro to Models
_written by Jaren N. Ashcraft_

`katsu` was originally written as a high-performance backend for functional Mueller calculus. This meant that we ignored "state" in lieu of developing the most rapid forward model we could for high-dimensional optimization. This choice resulted in difficult-to-customize code, so efforts were made to develop an object-oriented frontend for `katsu` to assist with fitting models to data.

This tutorial serves as an introduction to the `katsu.models` submodule, which provides a relatively lightweight and `jax`-able wrapper over `katsu.mueller` using `zodiax` to construct differentiable objects. 

Below is an example where we initialize a linear polarizer via the `LinearDiattenuator` class 

In [1]:
from katsu.models import LinearDiattenuator, LinearRetarder
from katsu.katsu_math import np, set_backend_to_jax


polarizer = LinearDiattenuator(transmission_axis=0, Tmin=0)

This object has a `matrix` attribute which stores the Mueller matrix computed in `katsu.mueller` 

In [2]:
polarizer.matrix

array([[0.5, 0.5, 0. , 0. ],
       [0.5, 0.5, 0. , 0. ],
       [0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. ]])

Thanks to zodiax, these objects work with our hot-swappable Jax backend. This needs to be done before initialization of the object, so unfortunately we do not support in-place data conversion.

In [3]:
set_backend_to_jax()
polarizer = LinearDiattenuator(transmission_axis=0, Tmin=0)
polarizer.matrix

Array([[0.5, 0.5, 0. , 0. ],
       [0.5, 0.5, 0. , 0. ],
       [0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. ]], dtype=float64)

Zodiax enables direct operations on our class attributes in a method reminiscent of operations on `jax` arrays. These include:
- `set`: which sets the new attribute
- arithmetic operations: `add`, `multiply`, `divide`, `power`
- simple analysis: `min`, `max`, `mean`

Use of these operations creates a new object with the updated attribute. See the appropriate use below for the `add` and `divide` methods

In [4]:
# Adding 1 to the matrix
polarizer = polarizer.add("matrix",1)
print(polarizer.matrix)
print(20*"-")

# subtracting 1 from the matrix
polarizer = polarizer.add("matrix", -1)
print(polarizer.matrix)
print(20*"-")

# Dividing by matrix element
polarizer = polarizer.divide("matrix", polarizer.matrix[0,0])
print(polarizer.matrix)
print(20*"-")

[[1.5 1.5 1.  1. ]
 [1.5 1.5 1.  1. ]
 [1.  1.  1.  1. ]
 [1.  1.  1.  1. ]]
--------------------
[[0.5 0.5 0.  0. ]
 [0.5 0.5 0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]]
--------------------
[[1. 1. 0. 0.]
 [1. 1. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
--------------------


## Alternative Constructors and Matrix Multiplication
Both the `LinearDiattenuator` and `LinearRetarder` come with alternative constructors for simple methods.

In [5]:
pol_horizontal = LinearDiattenuator.as_polarizer(0)
qwp_45degree = LinearRetarder.as_quarter_wave_plate(np.radians(45))

print(pol_horizontal.matrix)
print(qwp_45degree.matrix)

[[0.5 0.5 0.  0. ]
 [0.5 0.5 0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]]
[[ 1.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00]
 [ 0.000000e+00  6.123234e-17  6.123234e-17 -1.000000e+00]
 [ 0.000000e+00  6.123234e-17  1.000000e+00  6.123234e-17]
 [ 0.000000e+00  1.000000e+00 -6.123234e-17  6.123234e-17]]


These objects inherit from a `MuellerMatrix` base class which supports matrix multiplication of optics. This allows for using the `@` matmul operator between `MuellerMatrix` elements.

In [6]:
circular_pol = qwp_45degree @ pol_horizontal
print(circular_pol.matrix)

[[5.000000e-01 5.000000e-01 0.000000e+00 0.000000e+00]
 [3.061617e-17 3.061617e-17 0.000000e+00 0.000000e+00]
 [3.061617e-17 3.061617e-17 0.000000e+00 0.000000e+00]
 [5.000000e-01 5.000000e-01 0.000000e+00 0.000000e+00]]


## Full Models
To support optimization, we have a `Model` class that can simulate the power observed after a generally polarized source propagates through a system of Mueller matrices.

In [7]:
from katsu.models import Model

optics_list = [
    LinearDiattenuator.as_polarizer(0),
    LinearRetarder.as_quarter_wave_plate(np.radians(45))
]

model = Model(optics_list=optics_list)

AttributeError: 'LinearDiattenuator' object has no attribute 'variable'

TypeError: 'int' object is not subscriptable